# 03 — Vehicle Subset, Fold Assignment & Out-of-Sample Baselines

**Project:** EV Battery Capacity Prediction

Runs on artefacts from `01_data_ingestion.ipynb` and `02_eda.ipynb`.

---

## Purpose

This notebook produces the **evaluation protocol** that every model in this project —
baselines, LSTM and Transformer alike — will be measured under. Nothing downstream is
comparable unless it runs against the folds fixed here.

Three outputs:

1. A **30-vehicle subset**, selected to be representative rather than arbitrary
2. A **5-fold cross-validation assignment** at the vehicle level, saved to disk
3. **Out-of-sample baseline scores** — the definitive figures for the results table

## Why the baselines are recomputed here

The baseline figures in notebook 02 were computed **in-sample**: fitted on all 349,741
snippets and evaluated on those same snippets. The reference paper's LSTM (RMSE 1.420)
was evaluated on held-out data.

Comparing the two would be unfair to the LSTM. The figures below are computed the same
way the paper's were — fitted on training vehicles, evaluated on vehicles never seen —
and on **identical folds** to the ones the neural networks will use.

This last point is the important one. Given the 23× imbalance in per-vehicle snippet
counts (notebook 01, Finding A) and the distinct per-vehicle degradation trajectories
(notebook 02, Finding 4), a difference between two models evaluated on *different*
splits could easily be a difference in which vehicles happened to land in test.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
N_CARS = 30          # vehicles in the working subset
N_FOLDS = 5          # cross-validation folds

rng = np.random.default_rng(RANDOM_SEED)

INDEX_DIR = Path("../data/processed")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(INDEX_DIR / "dataset_index.json") as fp:
    index = json.load(fp)

car_to_files = index["car_to_files"]
cars_df = pd.read_csv(INDEX_DIR / "car_summary.csv")
snips = pd.read_csv(INDEX_DIR / "snippet_meta.csv")

print(f"Vehicles available : {len(cars_df)}")
print(f"Snippets available : {len(snips):,}")

## 1. Selecting 30 vehicles

### Why subset at all

All 100 vehicles give 349,741 snippets. With 5-fold CV, each architecture is trained
5 times — and during development each will be retrained many times over while tuning.
Roughly 105k snippets from 30 vehicles is ample for models of this size, and the
accuracy difference is small relative to the compute saved.

The full index is retained, so scaling up later is a one-line change.

### How to select them

Two properties from earlier findings constrain the choice:

**Cannot stratify by mileage.** (Notebook 01, Finding D.) Nearly every vehicle spans
low, medium and high mileage individually, so mileage is not a per-vehicle property to
balance on. Any reasonable set of vehicles will cover the mileage range.

**Should stratify by snippet count.** (Notebook 01, Finding A.) Counts range 390 to
9,101 — a 23× spread. Taking 30 vehicles at random risks landing on a cluster of
data-rich or data-poor ones, which would make the subset unrepresentative of the fleet.

So: sort vehicles by snippet count, divide into 30 equal-sized strata, and draw one
vehicle at random from each. This guarantees the subset spans the full range of data
volumes while remaining random within each stratum.

In [ ]:
# Stratified selection across snippet-count deciles.
# Sorting then splitting into N_CARS equal groups and drawing one from each
# guarantees coverage of the full 390-9,101 range of per-vehicle data volumes.

ordered = cars_df.sort_values("n_snippets").reset_index(drop=True)
strata = np.array_split(np.arange(len(ordered)), N_CARS)

picked_idx = [int(rng.choice(s)) for s in strata]
subset = ordered.iloc[picked_idx].sort_values("car").reset_index(drop=True)

print(f"Selected {len(subset)} vehicles")
print(f"Snippets : {subset['n_snippets'].sum():,} "
      f"({subset['n_snippets'].sum() / cars_df['n_snippets'].sum():.1%} of full set)")
print(f"Per-vehicle range: {subset['n_snippets'].min()} - "
      f"{subset['n_snippets'].max()}")
print()
print(subset[["car", "n_snippets", "label", "min_mileage", "max_mileage"]])

### Verify the subset is representative

Before committing, confirm the 30-vehicle subset preserves the distributional
properties established in notebook 02. In particular the **target standard deviation**,
since that sets the mean-baseline floor, and the **capacity minimum**, since the thin
left tail of degraded batteries is operationally the most important region.

In [ ]:
subset_cars = set(subset["car"])
sub_snips = snips[snips["car"].isin(subset_cars)].copy()

comp = pd.DataFrame({
    "full (100 cars)": [
        len(snips), snips["capacity"].mean(), snips["capacity"].std(),
        snips["capacity"].min(), snips["capacity"].max(),
        snips["mileage"].min() / 1000, snips["mileage"].max() / 1000,
        snips["mileage"].corr(snips["capacity"]),
    ],
    f"subset ({N_CARS} cars)": [
        len(sub_snips), sub_snips["capacity"].mean(), sub_snips["capacity"].std(),
        sub_snips["capacity"].min(), sub_snips["capacity"].max(),
        sub_snips["mileage"].min() / 1000, sub_snips["mileage"].max() / 1000,
        sub_snips["mileage"].corr(sub_snips["capacity"]),
    ],
}, index=["n_snippets", "capacity_mean", "capacity_std", "capacity_min",
          "capacity_max", "mileage_min_k", "mileage_max_k", "correlation"])

comp.round(3)

In [ ]:
# Side-by-side distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(snips["capacity"], bins=70, density=True, alpha=0.5,
             label="full (100 cars)", color="steelblue")
axes[0].hist(sub_snips["capacity"], bins=70, density=True, alpha=0.5,
             label=f"subset ({N_CARS} cars)", color="darkorange")
axes[0].set_xlabel("Capacity (Ah)"); axes[0].set_ylabel("Density")
axes[0].set_title("Target distribution"); axes[0].legend()

axes[1].hist(snips["mileage"] / 1000, bins=70, density=True, alpha=0.5,
             label="full", color="steelblue")
axes[1].hist(sub_snips["mileage"] / 1000, bins=70, density=True, alpha=0.5,
             label="subset", color="darkorange")
axes[1].set_xlabel("Mileage (thousand km)"); axes[1].set_ylabel("Density")
axes[1].set_title("Mileage coverage"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "subset_representativeness.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretation

*Fill in after running.* The checks that matter:

- **`capacity_std` close to 1.863** — if the subset were materially tighter or wider,
  the mean-baseline floor would shift and results would not be comparable to notebook 02
- **`capacity_min` near 26.64** — confirms degraded batteries survived the subsetting.
  If the minimum jumped to, say, 32 Ah, the subset would have lost the operationally
  important tail and should be re-drawn with a different seed
- **`correlation` near −0.800** — the degradation signal is intact
- **Overlapping histograms** — no systematic shift in either variable

## 2. Fold assignment

### Why k-fold rather than a single split

With 30 vehicles and an 80/20 split, the test set would be 6 vehicles. Given the 23×
imbalance in snippet counts and the distinct per-vehicle trajectories, the headline
number would depend heavily on *which* 6 vehicles landed in test — plausibly a larger
effect than the LSTM-vs-Transformer difference being measured.

The reference paper hit exactly this: a single vehicle accounted for roughly 70% of
their worst-error cases, and removing it moved their metrics by about 5%.

With 5-fold CV each vehicle appears in test exactly once, and results come with a
**standard deviation across folds** — which is what determines whether a gap between
two models is real or noise.

### Balancing the folds

Random fold assignment would produce uneven folds given the 23× spread in snippet
counts. Instead, vehicles are sorted by snippet count descending and each is assigned
to whichever fold currently holds the fewest snippets — a standard greedy load-balancing
heuristic that yields far more even folds than random assignment.

**Development uses fold 0 only.** Full 5-fold runs are reserved for final reported
results, so debugging and hyperparameter iteration stay fast.

In [ ]:
# Greedy balanced assignment: largest vehicles first, each to the lightest fold.

fold_of = {}
fold_load = {k: 0 for k in range(N_FOLDS)}

for _, row in subset.sort_values("n_snippets", ascending=False).iterrows():
    target = min(fold_load, key=fold_load.get)
    fold_of[int(row["car"])] = target
    fold_load[target] += int(row["n_snippets"])

subset["fold"] = subset["car"].map(fold_of)

fold_summary = subset.groupby("fold").agg(
    n_cars=("car", "count"),
    n_snippets=("n_snippets", "sum"),
    cars=("car", lambda s: sorted(s.tolist())),
)

print(fold_summary[["n_cars", "n_snippets"]])
print()
spread = fold_summary["n_snippets"].max() / fold_summary["n_snippets"].min()
print(f"Largest/smallest fold ratio: {spread:.2f}x  (1.00 would be perfectly even)")
print()
for f, row in fold_summary.iterrows():
    print(f"  fold {f}: {row['cars']}")

In [ ]:
# Per-fold target statistics — folds should be comparable, not just equal in size.

sub_snips["fold"] = sub_snips["car"].map(fold_of)

fold_stats = sub_snips.groupby("fold").agg(
    n=("capacity", "size"),
    cap_mean=("capacity", "mean"),
    cap_std=("capacity", "std"),
    cap_min=("capacity", "min"),
    mileage_mean_k=("mileage", lambda s: s.mean() / 1000),
).round(3)

fold_stats

### Interpretation

*Fill in after running.* What to look for:

- **Fold size ratio near 1.0** — the greedy assignment should keep folds within a few
  percent of each other
- **`cap_mean` similar across folds** — a fold whose vehicles happen to be unusually
  healthy or unusually worn will produce an outlying score for reasons unrelated to the
  model. Worth noting now so it can be referenced when interpreting per-fold results
- **`cap_std` similar across folds** — recall this sets each fold's mean-baseline floor,
  so a fold with lower std is intrinsically "easier" on RMSE

If one fold looks materially different, that is not necessarily a problem — it is
information to carry forward when reading the per-fold results table.

## 3. Save the fold manifest

Written to disk so the split is **fixed and reproducible**. Every subsequent notebook,
training script and evaluation run loads this file rather than re-deriving a split.

This is what makes the LSTM, Transformer and baselines genuinely comparable.

In [ ]:
manifest = {
    "config": {
        "n_cars": N_CARS,
        "n_folds": N_FOLDS,
        "random_seed": RANDOM_SEED,
        "selection": "stratified by snippet count, one vehicle per stratum",
        "fold_assignment": "greedy balanced by snippet count",
    },
    "folds": {
        str(f): sorted(int(c) for c in subset.loc[subset["fold"] == f, "car"])
        for f in range(N_FOLDS)
    },
    "car_to_fold": {str(k): int(v) for k, v in fold_of.items()},
    "files": {
        str(c): car_to_files[str(c)] for c in sorted(subset["car"])
    },
}

manifest_path = INDEX_DIR / "fold_manifest.json"
with open(manifest_path, "w") as fp:
    json.dump(manifest, fp, indent=2)

subset.to_csv(INDEX_DIR / "subset_cars.csv", index=False)

print(f"Saved → {manifest_path}")
print(f"Saved → {INDEX_DIR / 'subset_cars.csv'}")
print(f"\nTotal snippets in working set: "
      f"{sum(len(v) for v in manifest['files'].values()):,}")

## 4. Out-of-sample baselines

Both baselines, evaluated properly: fitted on the training vehicles of each fold and
scored on the held-out vehicles.

**Baseline 1 — mean predictor.** Predicts the *training* fold's mean for every test
snippet. Uses no input. Establishes the floor.

**Baseline 2 — mileage-only linear regression.** One feature: the odometer reading.
Fitted on training vehicles, predicts on unseen ones.

> Note that the mean baseline's RMSE will no longer exactly equal the target standard
> deviation, as it did in-sample. It predicts the *training* mean against the *test*
> distribution, so any difference between fold means adds error.

In [ ]:
def metrics(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
        "r2": float(r2_score(y_true, y_pred)),
    }


records = []

for fold in range(N_FOLDS):
    test_mask = sub_snips["fold"] == fold
    train, test = sub_snips[~test_mask], sub_snips[test_mask]

    y_train, y_test = train["capacity"].values, test["capacity"].values
    X_train = train[["mileage"]].values
    X_test = test[["mileage"]].values

    # --- mean predictor -----------------------------------------------------
    m = metrics(y_test, np.full_like(y_test, y_train.mean()))
    records.append({"model": "Mean baseline", "fold": fold, **m})

    # --- mileage-only linear regression -------------------------------------
    lr = LinearRegression().fit(X_train, y_train)
    m = metrics(y_test, lr.predict(X_test))
    records.append({"model": "Linear (mileage)", "fold": fold,
                    "slope_per_1k_km": float(lr.coef_[0] * 1000), **m})

per_fold = pd.DataFrame(records)
per_fold.round(3)

In [ ]:
# Aggregate: mean +/- std across folds. The std is what tells us whether a gap
# between two models is real or just fold-to-fold noise.

agg = per_fold.groupby("model")[["rmse", "mae", "mape", "r2"]].agg(["mean", "std"])
agg = agg.round(3)

print(agg)
print()

for model in per_fold["model"].unique():
    d = per_fold[per_fold["model"] == model]
    print(f"{model:<20} RMSE {d['rmse'].mean():.3f} ± {d['rmse'].std():.3f}   "
          f"(range {d['rmse'].min():.3f} - {d['rmse'].max():.3f})")

print(f"\nReference paper's LSTM (held out): RMSE 1.420, MAE 1.090, MAPE 2.70%")

In [ ]:
# Per-fold variation, visualised.
# A wide spread here is itself a finding: it quantifies how much a single-split
# result could have drifted purely by luck of the draw.

fig, ax = plt.subplots(figsize=(9, 4))

for model, marker in [("Mean baseline", "o"), ("Linear (mileage)", "s")]:
    d = per_fold[per_fold["model"] == model]
    ax.plot(d["fold"], d["rmse"], marker=marker, label=model)

ax.axhline(1.420, color="crimson", ls="--", lw=1,
           label="Reference paper LSTM (1.420)")
ax.set_xlabel("Fold"); ax.set_ylabel("RMSE (Ah)")
ax.set_xticks(range(N_FOLDS))
ax.set_title("Baseline RMSE by fold")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_by_fold.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretation

*Fill in after running.* The three things this establishes:

**1. The real bar.** The mileage baseline's mean RMSE is the number the LSTM and
Transformer must beat. Compare it to the in-sample 1.117 from notebook 02 — the
out-of-sample figure will be higher, because the model now faces vehicles whose
individual trajectories it has never seen.

**2. Whether it still beats the published benchmark.** If the out-of-sample mileage
baseline is still below 1.420, Finding 5 holds under fair comparison and the reframed
research question stands.

**3. How much fold-to-fold noise to expect.** The standard deviation across folds is
the key number. When the LSTM and Transformer results arrive, any difference between
them **smaller than this spread cannot be called a real difference**. Establishing it
now, before seeing model results, avoids the temptation to read significance into noise
later.

## 5. Baseline by mileage band

Notebook 02, Findings 3 and 3b predicted that the mileage baseline should be strongest
at low mileage (where every vehicle sits at 43.33 ± 0.78) and weakest at high mileage
(spread 1.38, only 4.1% of data).

Establishing the per-band baseline now gives the sequence models a **band-level**
reference, not just an aggregate one — which is where the hypothesised advantage should
appear if it exists at all.

In [ ]:
band_records = []
bins = [0, 50, 100, 150, 200, 300]

for fold in range(N_FOLDS):
    test_mask = sub_snips["fold"] == fold
    train, test = sub_snips[~test_mask], sub_snips[test_mask].copy()

    lr = LinearRegression().fit(train[["mileage"]].values, train["capacity"].values)
    test["pred"] = lr.predict(test[["mileage"]].values)
    test["band"] = pd.cut(test["mileage"] / 1000, bins=bins)

    for band, g in test.groupby("band", observed=True):
        if len(g) < 50:      # skip bands too small to be meaningful
            continue
        band_records.append({
            "band": str(band), "fold": fold, "n": len(g),
            **metrics(g["capacity"].values, g["pred"].values),
        })

bands = pd.DataFrame(band_records)

band_agg = bands.groupby("band").agg(
    folds=("fold", "nunique"),
    n_total=("n", "sum"),
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    mae_mean=("mae", "mean"),
).round(3)

band_agg

### Interpretation

*Fill in after running.* The expected pattern, per Finding 3:

- **Low mileage (0–50k):** the baseline should perform *well* — vehicles are uniform
  there, so knowing the odometer is nearly sufficient. Little room for a sequence model
  to add value.
- **High mileage (200k+):** the baseline should perform *worse* — vehicles have diverged
  and the odometer can only report a fleet average.

**This band table is the real target.** If the sequence models beat the odometer
anywhere, high-mileage bands are where it should show. Aggregate RMSE would dilute that
effect, since 41% of snippets sit in the single 100–150k band.

Watch the `n_total` column too: if the 200–300k band has very few test snippets per
fold, its RMSE will be noisy and conclusions drawn from it need appropriate caution.

In [ ]:
# Persist baseline results for the final comparison table.

RESULTS_DIR = Path("../reports")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

per_fold.to_csv(RESULTS_DIR / "baselines_per_fold.csv", index=False)
bands.to_csv(RESULTS_DIR / "baselines_per_band.csv", index=False)

print(f"Saved → {RESULTS_DIR / 'baselines_per_fold.csv'}")
print(f"Saved → {RESULTS_DIR / 'baselines_per_band.csv'}")

## Summary

**Configuration**

| Setting | Value |
|---|---|
| Vehicles | 30, stratified by snippet count |
| Folds | 5, balanced by snippet count |
| Split level | Vehicle (never snippet) |
| Seed | 42 |

**Artefacts written**

| File | Contents |
|---|---|
| `data/processed/fold_manifest.json` | Fold assignment + file paths per vehicle |
| `data/processed/subset_cars.csv` | The 30 selected vehicles with fold labels |
| `reports/baselines_per_fold.csv` | Out-of-sample baseline metrics per fold |
| `reports/baselines_per_band.csv` | Baseline metrics per mileage band |
| `reports/figures/subset_representativeness.png` | Subset vs full distributions |
| `reports/figures/baseline_by_fold.png` | Fold-to-fold RMSE variation |

**Results table — to be completed**

| Model | Inputs | RMSE | MAE | MAPE | R² |
|---|---|---|---|---|---|
| Mean baseline | none | — | — | — | — |
| Linear regression | mileage | — | — | — | — |
| LSTM | 128×8 sequence | | | | |
| Transformer | 128×8 sequence | | | | |
| *Reference paper LSTM* | *128×8 sequence* | *1.420* | *1.090* | *2.70%* | *—* |

---

**Next:** `04_dataset_and_lstm.ipynb` — a lazy-loading PyTorch `Dataset` reading from
the fold manifest, normalisation fitted on training folds only, and the LSTM baseline
reproducing the reference architecture.